# SpectrumEye — CNN Training v2 (Physics-Based Dataset)

**Before running:** `Runtime → Change runtime type → T4 GPU`

**Steps:**
1. Cell 1 — verify GPU
2. Cell 2 — install packages
3. Cell 3 — upload `ml_training_v2.zip`
4. Cell 4 — augment raw images (1 500 → 10 500)
5. Cell 5 — split into train / val / test
6. Cell 6 — verify dataset
7. Cell 7 — train (50 epochs, ~20–30 min on T4)
8. Cell 8 — preview results
9. Cell 9 — download model

**What changed vs v1:**
- Dataset is now physics-based (IQ simulation → STFT → fixed normalization)
- Zip contains raw images; augmentation and split run here on Colab
- 500 raw images × 3 classes × 7 augmentations = 10 500 training images

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow {tf.__version__}')
print(f'GPUs found: {len(gpus)}')

if not gpus:
    print('\nWARNING: No GPU detected.')
    print('Go to Runtime → Change runtime type → T4 GPU, then reconnect.')
else:
    print(f'GPU: {gpus[0].name}  ← ready')

In [ ]:
# ── Cell 2: Install packages ──────────────────────────────────────
# TensorFlow, NumPy, scipy, Pillow, scikit-learn are pre-installed.
# Only seaborn needs to be added.
!pip install seaborn -q
print('Packages ready.')

In [ ]:
# ── Cell 3: Upload and unzip ml_training_v2.zip ───────────────────
from google.colab import files
import zipfile, os

print('Select ml_training_v2.zip from your computer...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'\nUnzipping {zip_name}...')
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# Verify raw images arrived
CLASS_LABELS = ['Key_Signal', 'Walkie_Talkie', 'LTE']
print('\nRaw images found:')
for cls in CLASS_LABELS:
    path = f'/content/ml/dataset/raw/{cls}'
    n = len([f for f in os.listdir(path) if f.endswith('.png')]) if os.path.exists(path) else 0
    print(f'  {cls}: {n} images')

In [ ]:
# ── Cell 4: Augment raw images ────────────────────────────────────
# Applies 6 augmentations per image → 7× dataset size
# 1 500 raw → 10 500 augmented images
# Augmentations: time_shift, freq_shift, awgn, amplitude, noise_mix, time_flip
# Expected time: ~2–3 minutes

!python /content/ml/augment.py

In [ ]:
# ── Cell 5: Split into train / val / test ─────────────────────────
# Stratified split: 70% train / 15% val / 15% test
# Augmented variants of the same source image are always kept
# in the same split (no data leakage)

!python /content/ml/split_dataset.py

In [ ]:
# ── Cell 6: Verify dataset ────────────────────────────────────────
import os

CLASS_LABELS = ['Key_Signal', 'Walkie_Talkie', 'LTE']
total = 0

print('Dataset contents:\n')
for split in ['train', 'val', 'test']:
    split_total = 0
    for cls in CLASS_LABELS:
        path = f'/content/ml/dataset/{split}/{cls}'
        count = len([f for f in os.listdir(path) if f.endswith('.png')]) if os.path.exists(path) else 0
        split_total += count
        print(f'  {split}/{cls}: {count} images')
    print(f'  {split} total: {split_total}\n')
    total += split_total

print(f'Grand total: {total} images')

# Expected: ~7350 train, ~1575 val, ~1575 test = ~10500 total
if total < 9000:
    print('\nWARNING: fewer images than expected. Re-run cells 4 and 5.')
else:
    print('Dataset OK — ready to train.')

In [ ]:
# ── Cell 7: Train ─────────────────────────────────────────────────
# Full 50-epoch training with early stopping.
# Expected time on T4 GPU: 20–35 minutes.
# Outputs saved to /content/ml/models/v2_colab/

!python /content/ml/train.py \
    --epochs 50 \
    --batch  32 \
    --lr     1e-4 \
    --version v2_colab

In [ ]:
# ── Cell 8: Preview results ───────────────────────────────────────
from IPython.display import Image, display
import json, os

model_dir = '/content/ml/models/v2_colab'

# Classification report
report_path = f'{model_dir}/classification_report.txt'
if os.path.exists(report_path):
    print('=== Classification Report ===')
    print(open(report_path).read())

# Key metrics
config_path = f'{model_dir}/config.json'
if os.path.exists(config_path):
    cfg = json.load(open(config_path))
    print(f'Epochs run    : {cfg["epochs_run"]} / {cfg["epochs_max"]}')
    print(f'Test accuracy : {cfg["test_accuracy"]*100:.1f}%')

# Confusion matrix
cm_path = f'{model_dir}/confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(cm_path))

# Training curves
curves_path = f'{model_dir}/training_curves.png'
if os.path.exists(curves_path):
    display(Image(curves_path))

In [ ]:
# ── Cell 9: Download trained model ───────────────────────────────
# Downloads a zip containing:
#   best_model.keras, confusion_matrix.png, training_curves.png,
#   classification_report.txt, config.json

import shutil
from google.colab import files

output_zip = '/content/spectromeye_v2_colab'
shutil.make_archive(output_zip, 'zip', '/content/ml/models/v2_colab')

print('Downloading spectromeye_v2_colab.zip ...')
files.download(f'{output_zip}.zip')